# Federated experiments — SESSION A of 2

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # must precede torch import
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Not on Colab or already mounted:", e)

In [ ]:
PROJECT        = "/content/drive/MyDrive/Depression_FL_Project"
LABELS_CSV     = f"{PROJECT}/fl/labels_v1/fl_harmonized_labels.csv"
TEXT_EMBED_NPZ = f"{PROJECT}/output/PDCH/processed_v1/embeddings/pdch_text_emb_full_dialogue.npz"
AUDIO_TURN_DIR = f"{PROJECT}/audio/v1/turn_embeddings"
LEGACY_EEG_DIR = f"{PROJECT}/data/embeddings"                  # original (leaky) EEG embeddings, optional
SEARCHED_JSON  = f"{PROJECT}/fl/fusion_v1/fl_hparam_search_best.json"
R1_OUT         = f"{PROJECT}/fl/revision_r1"
PARTITIONS_CSV = f"{R1_OUT}/partitions.csv"
EEG_OOF_DIR    = f"{R1_OUT}/eeg_oof"
SESSION        = "A"
RUNS_CSV       = f"{R1_OUT}/runs_long.csv"     # each session writes ONLY its own file
LATEX_OUT      = f"{R1_OUT}/latex"

EEG_SOURCE       = "oof"
AUDIO_LAYER_MODE = "per_partition"  # "per_partition": WavLM layer re-selected by CV inside each training split
                                    # "fixed": layer 10 as in the paper (layer chosen on DAIC train+dev)
AUDIO_FIXED_LAYER = 10

N_SEEDS     = 3  # seeds per partition (model initialisation / mini-batch
TUNE_LR     = True                   # select LR per partition & method on pooled VAL macro-F1
LR_GRID     = [5e-4, 2e-4, 1e-4]     # candidate LRs (subset of the original LR_CANDIDATES)
TUNE_SEED   = 0                      # seed used for LR selection (its run is reused, not re-run)
PARTITIONS_TO_RUN = None             # None = all partitions in partitions.csv, or e.g. [0, 1, 2]

RUN_MAIN    = True                   # Section 7
RUN_FACTOR  = True                   # Section 8
RUN_EXPORT  = False  # set True ONLY after both sessions have finished
for p in (LABELS_CSV, TEXT_EMBED_NPZ, AUDIO_TURN_DIR, PARTITIONS_CSV, EEG_OOF_DIR):
    print(f"  exists={os.path.exists(p)}  {p}")
os.makedirs(LATEX_OUT, exist_ok=True)

In [ ]:
import glob, json, copy, random, time, itertools
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from scipy import stats
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

def set_all_seeds(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as e:
    print("[determinism] unavailable:", e)

## 3 · Hyper-parameters

In [ ]:
BASE_HP = dict(
    SHARED_DIM=512, LOCAL_EPOCHS=10, N_ROUNDS=100, EARLY_STOP_PATIENCE=20,
    LR_HEAD=1e-4, N_CLIENTS=3, HIDDEN_DIM=256, N_HEAD_LAYERS=2, HEAD_DROPOUT=0.2,
    USE_INPUT_STANDARDIZATION=True, BATCH_SIZE=32, AUGMENT_NOISE_STD=0.05,
    USE_CLASS_WEIGHTING=True, PROTO_TEMPERATURE=0.1, GRAD_CLIP=1.0,
    USE_SAMPLE_SIZE_WEIGHTING=False, USE_CLIENT_RELIABILITY_WEIGHTING=True,
    USE_COMPACTNESS=True, COMPACT_LAMBDA=0.20,
    USE_CLASSIFIER_HEAD=True, PREDICT_WITH="classifier", CE_WEIGHT=1.0,
    USE_FOCAL_LOSS=True, FOCAL_GAMMA=1.5, CE_LABEL_SMOOTHING=0.05,
    USE_MIXUP=True, MIXUP_ALPHA=0.2,
    USE_ETF_INIT=True, GATE_ADAPT_MOMENTUM=0.95, GATE_MIN=0.05, GATE_MAX=0.95,
    GATE_INIT={"audio": 0.0075, "eeg": 0.75, "text": 0.35},
    WEIGHT_CLIP_RANGE=(0.3, 1.5),
    USE_THRESHOLD_TUNING=True, THRESHOLD_GRID=41,
    TGP_MARGIN=0.5, TGP_LR=1e-3,
)
GATE_TARGET = {"audio": (0.01, 0.0075), "text": (0.60, 0.50), "eeg": (1.0, None)}

if os.path.exists(SEARCHED_JSON):
    _b = json.load(open(SEARCHED_JSON)); _bc = _b.get("best", {})
    for _k in ("LR_HEAD", "COMPACT_LAMBDA", "PROTO_TEMPERATURE", "AUGMENT_NOISE_STD",
               "HEAD_DROPOUT", "HIDDEN_DIM", "LOCAL_EPOCHS", "CE_WEIGHT", "FOCAL_GAMMA"):
        if _k in _bc:
            BASE_HP[_k] = int(_bc[_k]) if _k in ("HIDDEN_DIM", "LOCAL_EPOCHS") else _bc[_k]
    if "GATE_INIT" in _b:
        BASE_HP["GATE_INIT"] = {k: float(v) for k, v in _b["GATE_INIT"].items()}
    print("[hp] loaded searched values from", SEARCHED_JSON)
print(json.dumps({k: (v if not isinstance(v, tuple) else list(v)) for k, v in BASE_HP.items()}, indent=1))

def apply_config(task, cut=None):
    """Set task-dependent globals, then (re)define the core code (Section 4)."""
    g = globals()
    for k, v in copy.deepcopy(BASE_HP).items():
        g[k] = v
    g["TASK"] = task
    g["N_CLASSES"] = 2 if task == "binary" else 3
    g["BINARY_POSITIVE_FROM"] = cut
    exec(compile(CORE_SRC, "<core>", "exec"), g)

## 4 · Core FL code

In [ ]:
CORE_SRC = r'''
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd
import math
import matplotlib.pyplot as plt


def _safe_normalize(x, dim=-1, eps=1e-12):
    return x / (x.norm(dim=dim, keepdim=True).clamp_min(eps))

def make_simplex_etf(n_classes, dim, device=None, dtype=torch.float32):
    """Deterministic ETF / simplex initialization."""
    if n_classes < 2:
        raise ValueError("n_classes must be >= 2")
    if n_classes == 2:
        base = torch.tensor([[1.0], [-1.0]], dtype=dtype, device=device)
    elif n_classes == 3:
        base = torch.tensor([
            [1.0, 0.0],
            [-0.5,  math.sqrt(3) / 2.0],
            [-0.5, -math.sqrt(3) / 2.0],
        ], dtype=dtype, device=device)
    else:
        eye = torch.eye(n_classes, dtype=dtype, device=device)
        eye = eye - eye.mean(dim=0, keepdim=True)
        q, _ = torch.linalg.qr(eye.T, mode="reduced")
        base = q.T[:n_classes]
    if base.shape[1] < dim:
        pad = torch.zeros((n_classes, dim - base.shape[1]), dtype=dtype, device=device)
        base = torch.cat([base, pad], dim=1)
    elif base.shape[1] > dim:
        base = base[:, :dim]
    return _safe_normalize(base, dim=-1)

def _init_proto_bank(n_classes=N_CLASSES, dim=SHARED_DIM, use_etf=USE_ETF_INIT, device=None):
    if use_etf:
        return {c: make_simplex_etf(n_classes, dim, device=device)[c].detach().clone()
                for c in range(n_classes)}
    g = torch.randn(n_classes, dim, device=device)
    g = _safe_normalize(g, dim=-1)
    return {c: g[c].detach().clone() for c in range(n_classes)}

class ProjectionHead(nn.Module):
    """Native embedding dim -> shared 128-d."""
    def __init__(self, in_dim, out_dim=None, hidden_dim=None, n_layers=None, dropout=None):
        super().__init__()
        out_dim    = SHARED_DIM    if out_dim    is None else out_dim
        hidden_dim = HIDDEN_DIM    if hidden_dim is None else hidden_dim
        n_layers   = N_HEAD_LAYERS if n_layers   is None else n_layers
        dropout    = HEAD_DROPOUT  if dropout    is None else dropout
        layers = []
        d = in_dim
        for _ in range(n_layers):
            layers += [nn.Linear(d, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout)]
            d = hidden_dim
        layers += [nn.Linear(d, out_dim)]
        self.net = nn.Sequential(*layers)
        self.identity_path = (in_dim == out_dim)
        if USE_CLASSIFIER_HEAD:
            self.classifier = nn.Sequential(
                nn.Linear(out_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(hidden_dim, N_CLASSES))
        else:
            self.classifier = None

    def forward(self, x):
        out = self.net(x)
        if self.identity_path:
            out = 0.5 * x + 0.5 * out
        return F.normalize(out, dim=-1)

    def classify(self, emb):
        """Logits from the projected (already L2-normalized) representation."""
        return self.classifier(emb)

def compute_sample_size_weights(client_data, clip_range=WEIGHT_CLIP_RANGE):
    """Kept for ablation only."""
    sizes = {silo: len(y) for silo, (X, y) in client_data.items()}
    total = sum(sizes.values())
    n_clients = len(sizes)
    raw = {silo: (total / (n_clients * n)) for silo, n in sizes.items()}
    lo, hi = clip_range
    return {silo: float(np.clip(w, lo, hi)) for silo, w in raw.items()}

def compute_local_prototypes(emb, y, n_classes=N_CLASSES):
    protos = {}
    for c in range(n_classes):
        mask = (y == c)
        n = int(mask.sum().item())
        if n > 0:
            protos[c] = (F.normalize(emb[mask].mean(dim=0).detach(), dim=-1), n)
    return protos

def aggregate_prototypes_naive(client_protos, n_classes=N_CLASSES):
    """Vanilla FedProto: weighted-average by sample count."""
    g = {}
    for c in range(n_classes):
        vecs, w = [], []
        for cp in client_protos:
            if c in cp:
                v, n = cp[c]
                vecs.append(F.normalize(v, dim=-1))
                w.append(n)
        if vecs:
            w = torch.tensor(w, dtype=torch.float32)
            w = w / w.sum().clamp_min(1e-12)
            g[c] = _safe_normalize((torch.stack(vecs) * w.unsqueeze(-1)).sum(dim=0), dim=-1)
    return g

def aggregate_prototypes_reliability(
    client_protos,
    client_reliability,
    n_classes=N_CLASSES,
    eps=1e-8
):
    """Reliability-weighted prototype aggregation."""

    g = {}

    silo_names = list(client_reliability.keys())

    for c in range(n_classes):

        vecs = []
        weights = []

        for idx, cp in enumerate(client_protos):

            if c not in cp:
                continue

            proto, _ = cp[c]

            if idx < len(silo_names):
                silo_name = silo_names[idx]

                rel = float(
                    client_reliability
                    .get(silo_name, {})
                    .get(c, 0.0)
                )
            else:
                rel = 0.0

            rel = max(rel, eps)

            vecs.append(
                F.normalize(proto, dim=-1)
            )

            weights.append(rel)

        if len(vecs) == 0:
            continue

        weights = torch.tensor(
            weights,
            dtype=torch.float32,
            device=vecs[0].device
        )

        weights = (
            weights /
            weights.sum().clamp_min(eps)
        )

        g[c] = _safe_normalize(
            (
                torch.stack(vecs)
                * weights.unsqueeze(-1)
            ).sum(dim=0),
            dim=-1
        )

    return g

class TrainableGlobalPrototypes(nn.Module):
    """FedTGP-style trainable global prototypes, now ETF-initialized and kept on the hypersphere."""
    def __init__(self, n_classes=N_CLASSES, dim=SHARED_DIM, margin=0.5, init_prototypes=None):
        super().__init__()
        if init_prototypes is None:
            init_prototypes = _init_proto_bank(n_classes=n_classes, dim=dim, use_etf=True)
        init = torch.stack([init_prototypes[c] for c in range(n_classes)], dim=0)
        self.prototypes = nn.Parameter(F.normalize(init, dim=-1))
        self.margin = margin

    def contrastive_update_loss(self, client_protos):
        losses = []
        g = F.normalize(self.prototypes, dim=-1)
        for cp in client_protos:
            for c, (local_proto, _n) in cp.items():
                local_proto = F.normalize(local_proto, dim=-1)
                pos = F.cosine_similarity(local_proto.unsqueeze(0), g[c].unsqueeze(0))
                negs = torch.stack([F.cosine_similarity(local_proto.unsqueeze(0), g[k].unsqueeze(0))
                                    for k in range(g.shape[0]) if k != c])
                losses.append(F.relu(self.margin - (pos - negs.mean())))
        return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=g.device, requires_grad=True)

    def as_dict(self):
        g = F.normalize(self.prototypes, dim=-1).detach()
        return {c: g[c] for c in range(g.shape[0])}

def fedproto_local_loss(emb, y, global_protos, temperature=None, class_weights=None):
    """Prototypical contrastive loss: pull toward own-class global prototype, push from others."""
    temperature = PROTO_TEMPERATURE if temperature is None else temperature
    if not global_protos:
        return torch.tensor(0.0, device=emb.device, requires_grad=True)

    classes_present = sorted(global_protos.keys())
    if len(classes_present) < 2:
        return torch.tensor(0.0, device=emb.device, requires_grad=True)

    proto_mat = torch.stack([F.normalize(global_protos[c], dim=-1) for c in classes_present])
    losses = []
    for i in range(emb.shape[0]):
        yi = int(y[i].item())
        if yi not in global_protos:
            continue
        sims = F.cosine_similarity(F.normalize(emb[i].unsqueeze(0), dim=-1), proto_mat) / temperature
        target = classes_present.index(yi)
        ce = F.cross_entropy(sims.unsqueeze(0), torch.tensor([target], device=emb.device))
        if class_weights is not None:
            ce = ce * class_weights[yi]
        losses.append(ce)
    return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=emb.device, requires_grad=True)

def intra_class_compactness_loss(emb, y, temperature=0.07):
    emb = F.normalize(emb, dim=-1)
    sim = emb @ emb.T
    losses = []
    for i in range(len(y)):
        pos_mask = (y == y[i])
        pos_mask[i] = False
        neg_mask = (y != y[i])
        if pos_mask.sum() == 0:
            continue
        pos = sim[i][pos_mask] / temperature
        neg = sim[i][neg_mask] / temperature
        numerator = torch.exp(pos).sum()
        denominator = numerator + torch.exp(neg).sum()
        losses.append(-torch.log(numerator / denominator))
    if len(losses) == 0:
        return torch.tensor(0.0, device=emb.device, requires_grad=True)
    return torch.stack(losses).mean()

def classification_loss(logits, y, class_weights=None):
    """Class-balanced classifier loss."""
    if USE_FOCAL_LOSS:
        logp = F.log_softmax(logits, dim=-1)
        ce = F.nll_loss(logp, y, weight=class_weights, reduction='none')
        pt = logp.gather(1, y.unsqueeze(1)).squeeze(1).exp()
        return (((1.0 - pt) ** FOCAL_GAMMA) * ce).mean()
    return F.cross_entropy(logits, y, weight=class_weights, label_smoothing=CE_LABEL_SMOOTHING)

def _mixup_ce(head, xb, yb, class_weights=None):
    """Mixup on the classification path only (proto/compact losses keep hard labels)."""
    if (not USE_MIXUP) or MIXUP_ALPHA <= 0 or xb.shape[0] < 2:
        return classification_loss(head.classify(head(xb)), yb, class_weights)
    lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))
    perm = torch.randperm(xb.shape[0], device=xb.device)
    xmix = lam * xb + (1.0 - lam) * xb[perm]
    logits = head.classify(head(xmix))
    return (lam * classification_loss(logits, yb, class_weights)
            + (1.0 - lam) * classification_loss(logits, yb[perm], class_weights))

def positive_prob(head, emb):
    """P(class=1) from the classifier head on a projected (normalized) embedding."""
    return torch.softmax(head.classify(emb), dim=-1)[:, 1]

def best_threshold_on_val(head, Xval, yval):
    """Binary only: choose the decision threshold that maximizes macro-F1 on this silo's VAL set."""
    if N_CLASSES != 2 or not USE_THRESHOLD_TUNING \
       or PREDICT_WITH != 'classifier' or getattr(head, 'classifier', None) is None:
        return 0.5
    head.eval()
    with torch.no_grad():
        p = positive_prob(head, head(Xval)).cpu().numpy()
    yv = yval.cpu().numpy() if torch.is_tensor(yval) else np.asarray(yval)
    best_t, best_f1 = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, THRESHOLD_GRID):
        f1 = f1_score(yv, (p >= t).astype(int), average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t

def predict_labels(head, emb, global_protos, pos_threshold=None):
    """Unified prediction: trained classifier head if enabled, else nearest-prototype."""
    if PREDICT_WITH == 'classifier' and getattr(head, 'classifier', None) is not None:
        if N_CLASSES == 2 and pos_threshold is not None:
            return (positive_prob(head, emb) >= pos_threshold).long()
        return head.classify(emb).argmax(dim=-1)
    return predict_via_prototypes(emb, global_protos)

def predict_via_prototypes(emb, global_protos):
    classes_present = sorted(global_protos.keys())
    if not classes_present:
        return torch.zeros(len(emb), dtype=torch.long, device=emb.device)
    proto_mat = torch.stack([F.normalize(global_protos[c], dim=-1) for c in classes_present])
    sims = F.normalize(emb, dim=-1) @ F.normalize(proto_mat, dim=-1).T
    idx = sims.argmax(dim=-1)
    return torch.tensor([classes_present[i] for i in idx.tolist()], device=emb.device)

def per_class_reliability(y_true, y_pred, n_classes=N_CLASSES):
    scores = f1_score(
        y_true,
        y_pred,
        average=None,
        labels=list(range(n_classes)),
        zero_division=0
    )
    return {c: float(scores[c]) for c in range(n_classes)}

def class_weights_from_labels(y, n_classes=N_CLASSES, eps=1e-6):
    """Inverse-frequency class weights (present classes only; absent -> 0)."""
    counts = torch.bincount(y, minlength=n_classes).float()
    present = counts > 0
    w = torch.zeros(n_classes)
    w[present] = 1.0 / counts[present]
    w[present] = w[present] * (present.sum().float() / w[present].sum().clamp_min(eps))
    return w

def run_one_round(
    tdata,
    heads,
    global_protos,
    optimizers,
    local_epochs=None,
    trainable_global=None,
    client_gates=None,
    warmup=False,
    use_reliability_weighting=None,
    grad_clip=None
):
    # live-resolve so a search loop can vary these via globals
    local_epochs = LOCAL_EPOCHS if local_epochs is None else local_epochs
    use_reliability_weighting = USE_CLIENT_RELIABILITY_WEIGHTING if use_reliability_weighting is None else use_reliability_weighting
    grad_clip = GRAD_CLIP if grad_clip is None else grad_clip

    client_local_protos = []
    client_reliability = {}
    round_losses = []

    for silo in tdata:

        Xtr, ytr = tdata[silo]["train"]
        Xva, yva = tdata[silo]["val"]

        head = heads[silo]
        opt = optimizers[silo]

        gate = (
            client_gates.get(silo, 0.5)
            if client_gates is not None
            else 0.5
        )

        head.train()
        cw = class_weights_from_labels(ytr) if USE_CLASS_WEIGHTING else None
        n_train = Xtr.shape[0]

        for _ in range(local_epochs):

            perm = torch.randperm(n_train)

            for start in range(0, n_train, BATCH_SIZE):

                idx = perm[start:start + BATCH_SIZE]
                xb = Xtr[idx]
                yb = ytr[idx]

                # cheap augmentation: jitter embeddings to expand effective sample size
                if AUGMENT_NOISE_STD > 0:
                    xb = xb + AUGMENT_NOISE_STD * torch.randn_like(xb)

                opt.zero_grad()

                emb = head(xb)

                proto_loss = (
                    fedproto_local_loss(emb, yb, global_protos, class_weights=cw)
                    if len(global_protos) > 0
                    else torch.tensor(0.0, device=emb.device, requires_grad=True)
                )

                if USE_COMPACTNESS:
                    compact_loss = intra_class_compactness_loss(emb, yb)
                else:
                    compact_loss = torch.tensor(0.0, device=emb.device, requires_grad=True)

                # FedPer-style supervised classification term (the headline predictor).
                if USE_CLASSIFIER_HEAD and head.classifier is not None:
                    ce_loss = _mixup_ce(head, xb, yb, class_weights=cw)
                else:
                    ce_loss = torch.tensor(0.0, device=emb.device, requires_grad=True)

                total_loss = gate * proto_loss + COMPACT_LAMBDA * compact_loss + CE_WEIGHT * ce_loss

                total_loss.backward()

                torch.nn.utils.clip_grad_norm_(head.parameters(), grad_clip)

                opt.step()

        head.eval()

        with torch.no_grad():

            emb_train = head(Xtr)

            local_proto = compute_local_prototypes(
                emb_train,
                ytr
            )

            client_local_protos.append(
                local_proto
            )

            emb_val = head(Xva)

            pred_val = predict_via_prototypes(
                emb_val,
                global_protos
            )

            client_reliability[silo] = (
                per_class_reliability(
                    yva.cpu().numpy(),
                    pred_val.cpu().numpy(),
                    N_CLASSES
                )
            )

        round_losses.append(
            float(total_loss.item())
        )

    # Global prototype update
    if trainable_global is not None:

        proto_optimizer = torch.optim.Adam(
            trainable_global.parameters(),
            lr=1e-3
        )

        proto_optimizer.zero_grad()

        gp_loss = (
            trainable_global
            .contrastive_update_loss(
                client_local_protos
            )
        )

        gp_loss.backward()

        proto_optimizer.step()

        with torch.no_grad():
            trainable_global.prototypes.data = (
                F.normalize(
                    trainable_global.prototypes.data,
                    dim=-1
                )
            )

        new_global = (
            trainable_global.as_dict()
        )

    else:

        if (
            use_reliability_weighting
            and len(client_reliability) > 0
        ):

            new_global = (
                aggregate_prototypes_reliability(
                    client_local_protos,
                    client_reliability
                )
            )

        else:

            new_global = (
                aggregate_prototypes_naive(
                    client_local_protos
                )
            )

    return (
        new_global,
        client_local_protos,
        float(np.mean(round_losses)),
        client_reliability
    )

print("Core FL functions defined (ETF init + warm-up + reliability-weighted aggregation enabled).")
'''
print("core source:", len(CORE_SRC), "chars")

## 5 · Training loop with a matched local control

In [ ]:
def to_tensor_data(splits):
    out = {}
    for silo, (tr, va, te) in splits.items():
        Xtr, ytr = tr; Xva, yva = va; Xte, yte = te
        if USE_INPUT_STANDARDIZATION:
            mu = Xtr.mean(axis=0, keepdims=True); sd = Xtr.std(axis=0, keepdims=True) + 1e-6
            Xtr, Xva, Xte = (Xtr - mu) / sd, (Xva - mu) / sd, (Xte - mu) / sd
        out[silo] = {k: (torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
                     for k, (X, y) in zip(("train", "val", "test"), ((Xtr, ytr), (Xva, yva), (Xte, yte)))}
    return out

def silo_val_macro_f1(head, protos, d):
    head.eval()
    with torch.no_grad():
        pred = predict_labels(head, head(d["val"][0]), protos)
    return f1_score(d["val"][1].numpy(), pred.cpu().numpy(), average="macro", zero_division=0)

def run_fl_training_r1(tdata, in_dims, seed, mode="fedproto", lr=None):
    assert mode in ("fedproto", "fedtgp", "local_proto", "local_tgp")
    set_all_seeds(seed)
    lr = LR_HEAD if lr is None else lr
    heads = {s: ProjectionHead(in_dims[s]) for s in tdata}
    optimizers = {s: torch.optim.AdamW(h.parameters(), lr=lr, weight_decay=1e-4) for s, h in heads.items()}
    use_tgp = mode in ("fedtgp", "local_tgp")
    groups = [tuple([s]) for s in tdata] if mode.startswith("local") else [tuple(tdata)]
    tgp, protos = {}, {}
    for g in groups:
        init = _init_proto_bank(N_CLASSES, SHARED_DIM, use_etf=USE_ETF_INIT)
        tgp[g] = (TrainableGlobalPrototypes(n_classes=N_CLASSES, dim=SHARED_DIM, margin=TGP_MARGIN,
                                            init_prototypes=init) if use_tgp else None)
        protos[g] = tgp[g].as_dict() if use_tgp else init
    group_of = {s: g for g in groups for s in g}
    client_gates = {s: float(GATE_INIT[s]) for s in tdata}

    best_val, best_round, best_state, best_protos, no_improve, hist = -1.0, -1, None, None, 0, []
    for r in range(N_ROUNDS):
        client_reliability, losses = {}, []
        for g in groups:
            sub = {s: tdata[s] for s in g}
            protos[g], _, loss, rel = run_one_round(
                tdata=sub, heads=heads, global_protos=protos[g], optimizers=optimizers,
                trainable_global=tgp[g], client_gates=client_gates,
                use_reliability_weighting=USE_CLIENT_RELIABILITY_WEIGHTING)
            client_reliability.update(rel); losses.append(loss)
        if USE_CLIENT_RELIABILITY_WEIGHTING:                       # adaptive gate (paper Eq. 9)
            for s in client_gates:
                vals = list(client_reliability.get(s, {}).values())
                rel = float(np.clip(np.mean(vals) if vals else 0.5, GATE_MIN, GATE_MAX))
                mult, cap = GATE_TARGET[s]
                target = rel * mult if cap is None else min(rel * mult, cap)
                client_gates[s] = GATE_ADAPT_MOMENTUM * client_gates[s] + (1 - GATE_ADAPT_MOMENTUM) * target
        silo_protos = {s: protos[group_of[s]] for s in tdata}
        val = float(np.mean([silo_val_macro_f1(heads[s], silo_protos[s], tdata[s]) for s in tdata]))
        hist.append({"round": r, "loss": float(np.mean(losses)), "val": val,
                     **{f"gate_{s}": client_gates[s] for s in client_gates}})
        if val > best_val:
            best_val, best_round, no_improve = val, r, 0
            best_state = {s: {k: v.clone() for k, v in h.state_dict().items()} for s, h in heads.items()}
            best_protos = {s: {c: v.clone() for c, v in silo_protos[s].items()} for s in tdata}
        else:
            no_improve += 1
            if no_improve >= EARLY_STOP_PATIENCE:
                break
    for s, h in heads.items():
        h.load_state_dict(best_state[s])
    return heads, best_protos, best_round, best_val, pd.DataFrame(hist)

def evaluate_test(heads, silo_protos, tdata):
    out = {}
    for s in tdata:
        heads[s].eval()
        thr = best_threshold_on_val(heads[s], tdata[s]["val"][0], tdata[s]["val"][1]) if N_CLASSES == 2 else None
        with torch.no_grad():
            pred = predict_labels(heads[s], heads[s](tdata[s]["test"][0]), silo_protos[s], pos_threshold=thr)
        yt, yp = tdata[s]["test"][1].numpy(), pred.cpu().numpy()
        out[s] = dict(f1=f1_score(yt, yp, average="macro", zero_division=0), thr=thr,
                      y=yt.tolist(), p=yp.tolist())
    return out

def standalone_probe(splits, seed=0):
    out = {}
    for s, (tr, va, te) in splits.items():
        sc = StandardScaler().fit(tr[0])
        clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed).fit(sc.transform(tr[0]), tr[1])
        yp = clf.predict(sc.transform(te[0]))
        out[s] = dict(f1=f1_score(te[1], yp, average="macro", zero_division=0), thr=None,
                      y=te[1].tolist(), p=yp.tolist())
    return out

## 6 · Data per partition, endpoints

In [ ]:
ENDPOINTS = {
    "bin_ea":    dict(task="binary",    silos=["eeg", "audio"],         cut={"eeg": 1, "audio": 1}),
    # consistent endpoints across all three silos
    "bin_t1":    dict(task="binary",    silos=["eeg", "audio", "text"], cut={"eeg": 1, "audio": 1, "text": 1}),
    "bin_t2":    dict(task="binary",    silos=["eeg", "audio", "text"], cut={"eeg": 2, "audio": 2, "text": 2}),
    "sev3":      dict(task="severity3", silos=["eeg", "audio", "text"], cut=None),
    "bin_mixed": dict(task="binary",    silos=["eeg", "audio", "text"], cut={"eeg": 1, "audio": 1, "text": 2}),
}
MAIN_ENDPOINTS = ["bin_ea", "bin_t1", "bin_t2", "sev3"]          # add "bin_mixed" if wanted
MAIN_METHODS   = ["standalone", "local_proto", "fedproto", "local_tgp", "fedtgp"]

parts = pd.read_csv(PARTITIONS_CSV, dtype={"subject_id": str})
ALL_PARTITIONS = sorted(parts.partition.unique()) if PARTITIONS_TO_RUN is None else list(PARTITIONS_TO_RUN)
print("partitions:", ALL_PARTITIONS)

# ---- raw embeddings ----
_t = np.load(TEXT_EMBED_NPZ, allow_pickle=True)
TEXT_RAW = dict(zip([str(i) for i in _t["ids"]], _t["X"].astype(np.float32)))

AUDIO_LAYERS = {}                                   # pid -> (13, 1536) mean+std per WavLM layer
for f in sorted(glob.glob(f"{AUDIO_TURN_DIR}/*.npz")):
    emb = np.load(f)["emb"]                         # (n_turns, 13, 768)
    AUDIO_LAYERS[str(int(os.path.basename(f).split(".")[0]))] = np.concatenate(
        [emb.mean(axis=0), emb.std(axis=0)], axis=1).astype(np.float32)
print(f"text={len(TEXT_RAW)} audio={len(AUDIO_LAYERS)}")

def eeg_raw(partition):
    if EEG_SOURCE == "oof":
        d = np.load(f"{EEG_OOF_DIR}/eeg_emb_partition{partition}.npz", allow_pickle=True)
        return dict(zip([str(i) for i in d["ids"]], d["X"]))
    Xs, ids = [], []
    for sp in ("train", "val", "test"):
        Xs.append(np.load(f"{LEGACY_EEG_DIR}/eeg_embeddings_{sp}.npy"))
        ids.append(np.load(f"{LEGACY_EEG_DIR}/eeg_subj_ids_{sp}.npy", allow_pickle=True))
    return dict(zip([str(i) for i in np.concatenate(ids)], np.concatenate(Xs)))

_audio_layer_cache = {}
def audio_layer_for(partition):
    """WavLM layer chosen inside the training split only (5-fold CV of the LR probe,..."""
    if AUDIO_LAYER_MODE == "fixed":
        return AUDIO_FIXED_LAYER
    if partition in _audio_layer_cache:
        return _audio_layer_cache[partition]
    P = parts[(parts.silo == "audio") & (parts.partition == partition) & (parts.split == "train")]
    y = (P.y_unified.to_numpy() >= 1).astype(int)
    scores = []
    for L in range(13):
        X = np.stack([AUDIO_LAYERS[s][L] for s in P.subject_id])
        Xs = StandardScaler().fit_transform(X)   # inside-CV scaling difference is negligible for selection
        pred = cross_val_predict(LogisticRegression(max_iter=2000, class_weight="balanced"), Xs, y,
                                 cv=StratifiedKFold(5, shuffle=True, random_state=partition))
        scores.append(f1_score(y, pred, average="macro"))
    _audio_layer_cache[partition] = int(np.argmax(scores))
    return _audio_layer_cache[partition]

def build_splits(endpoint, partition, balance="natural"):
    ep = ENDPOINTS[endpoint]
    raw = {"eeg": eeg_raw(partition), "text": TEXT_RAW,
           "audio": {k: v[audio_layer_for(partition)] for k, v in AUDIO_LAYERS.items()}}
    splits = {}
    for s in ep["silos"]:
        P = parts[(parts.silo == s) & (parts.partition == partition)]
        trio = []
        for sp in ("train", "val", "test"):
            Q = P[P.split == sp]
            y = Q.y_unified.to_numpy().astype(np.int64)
            if ep["task"] == "binary":
                y = (y >= ep["cut"][s]).astype(np.int64)
            X = np.stack([raw[s][sid] for sid in Q.subject_id]).astype(np.float32)
            if sp == "train" and balance == "balanced":          # undersample TRAIN only
                rng = np.random.RandomState(10_000 + partition)
                n_min = np.bincount(y).min()
                keep = np.concatenate([rng.choice(np.where(y == c)[0], n_min, replace=False)
                                       for c in np.unique(y)])
                X, y = X[np.sort(keep)], y[np.sort(keep)]
            trio.append((X, y))
        splits[s] = tuple(trio)
    return splits

## 7 · Runner (resumable) and main experiments

In [ ]:
RUN_COLS = ["study", "endpoint", "balance", "partition", "method", "lr", "seed", "silo",
            "test_f1", "val_pooled", "best_round", "thr", "audio_layer", "n_train", "n_test", "y", "p"]
done = pd.read_csv(RUNS_CSV) if os.path.exists(RUNS_CSV) else pd.DataFrame(columns=RUN_COLS)

def _have(study, endpoint, balance, partition, method, lr, seed):
    if done.empty:
        return False
    m = ((done.study == study) & (done.endpoint == endpoint) & (done.balance == balance) &
         (done.partition == partition) & (done.method == method) & (done.seed == seed))
    m &= done.lr.isna() if lr is None else np.isclose(done.lr.astype(float), lr)
    return bool(m.any())

def _append(rows):
    global done
    df = pd.DataFrame(rows, columns=RUN_COLS)
    df.to_csv(RUNS_CSV, mode="a", header=not os.path.exists(RUNS_CSV), index=False)
    done = pd.concat([done, df], ignore_index=True)

def run_cell(study, endpoint, partition, methods, balance="natural"):
    ep = ENDPOINTS[endpoint]
    apply_config(ep["task"], ep["cut"])
    splits = build_splits(endpoint, partition, balance)
    tdata = to_tensor_data(splits)
    in_dims = {s: tdata[s]["train"][0].shape[1] for s in tdata}
    layer = audio_layer_for(partition) if "audio" in ep["silos"] else -1
    meta = lambda s: dict(audio_layer=layer, n_train=len(splits[s][0][1]), n_test=len(splits[s][2][1]))

    if "standalone" in methods and not _have(study, endpoint, balance, partition, "standalone", None, -1):
        res = standalone_probe(splits)
        _append([dict(study=study, endpoint=endpoint, balance=balance, partition=partition,
                      method="standalone", lr=None, seed=-1, silo=s, test_f1=r["f1"], val_pooled=np.nan,
                      best_round=-1, thr=None, y=json.dumps(r["y"]), p=json.dumps(r["p"]), **meta(s))
                 for s, r in res.items()])

    for method in [m for m in methods if m != "standalone"]:
        def one(lr, seed):
            if _have(study, endpoint, balance, partition, method, lr, seed):
                return
            t0 = time.time()
            heads, protos, rnd, val, _ = run_fl_training_r1(tdata, in_dims, seed, mode=method, lr=lr)
            res = evaluate_test(heads, protos, tdata)
            _append([dict(study=study, endpoint=endpoint, balance=balance, partition=partition,
                          method=method, lr=lr, seed=seed, silo=s, test_f1=r["f1"], val_pooled=val,
                          best_round=rnd, thr=r["thr"], y=json.dumps(r["y"]), p=json.dumps(r["p"]), **meta(s))
                     for s, r in res.items()])
            print(f"  [{study}|{endpoint}|{balance}|p{partition}|{method}|lr={lr:g}|s{seed}] "
                  + " ".join(f"{s}={r['f1']:.3f}" for s, r in res.items()) + f" | val={val:.3f} | {time.time()-t0:.0f}s")
        grid = LR_GRID if TUNE_LR else [BASE_HP["LR_HEAD"]]
        for lr in grid:
            one(lr, TUNE_SEED)
        sel = done[(done.study == study) & (done.endpoint == endpoint) & (done.balance == balance) &
                   (done.partition == partition) & (done.method == method) & (done.seed == TUNE_SEED)]
        sel = sel[sel.lr.astype(float).apply(lambda v: any(np.isclose(v, g) for g in grid))]
        best_lr = float(sel.groupby("lr").val_pooled.first().idxmax())
        for seed in range(N_SEEDS):
            one(best_lr, seed)

In [ ]:
torch.set_grad_enabled(True)  # safety: re-enable gradients if an earlier stop left
SESSION_MAIN_ENDPOINTS = ["bin_ea", "bin_t1", "bin_t2"]   # SESSION A share of the main experiments
if RUN_MAIN:
    for endpoint in SESSION_MAIN_ENDPOINTS:
        for r in ALL_PARTITIONS:
            run_cell("main", endpoint, r, MAIN_METHODS)
    print("SESSION A main done")

## 8 · Controlled factor experiment

In [ ]:
torch.set_grad_enabled(True)
FACTOR_ENDPOINTS = ["sev3", "bin_t1", "bin_t2"]
FACTOR_METHODS = ["local_proto", "fedproto"]
SESSION_FACTOR_ENDPOINTS = ["bin_t1"]   # SESSION A share of the balanced cells
if RUN_FACTOR:
    for endpoint in SESSION_FACTOR_ENDPOINTS:
        for r in ALL_PARTITIONS:
            run_cell("factor", endpoint, r, FACTOR_METHODS, balance="balanced")
    print("SESSION A factor done")

## 9 · Statistics and LaTeX export

In [ ]:
import json, os
import numpy as np, pandas as pd
from scipy import stats

EP_CODE = {"bin_ea": "BinEA", "bin_t1": "BinOne", "bin_t2": "BinTwo", "sev3": "Sev", "bin_mixed": "BinMix"}
EP_SHORT = {"sev3": "3-class", "bin_t1": r"Binary $\theta{=}1$", "bin_t2": r"Binary $\theta{=}2$"}
EP_NAME = {"bin_ea": r"Binary, EEG + audio (depressed vs.\ not)",
           "bin_t1": r"Binary, all silos, $\theta=1$ (depressed vs.\ not)",
           "bin_t2": r"Binary, all silos, $\theta=2$ (severe vs.\ non-severe)",
           "sev3": "Three-class severity",
           "bin_mixed": r"Binary, original mixed endpoint"}
M_CODE = {"standalone": "Stand", "local_proto": "Loc", "fedproto": "FP", "local_tgp": "LocT", "fedtgp": "TGP"}
M_NAME = {"standalone": "Standalone probe", "local_proto": "Local (matched to FedProto)",
          "fedproto": "FedProto-style", "local_tgp": "Local (matched to FedTGP)", "fedtgp": "FedTGP-style"}
M_SHORT = {"standalone": "Standalone", "local_proto": "Local$_{\\mathrm{P}}$", "fedproto": "FedProto",
           "local_tgp": "Local$_{\\mathrm{T}}$", "fedtgp": "FedTGP"}
S_CODE = {"eeg": "EEG", "audio": "Aud", "text": "Txt", "all": "All"}
S_NAME = {"eeg": "EEG", "audio": "Audio", "text": "Text", "all": "Overall"}
PAIRS = [("fedproto", "local_proto", "FPvLoc"), ("fedtgp", "local_tgp", "TGPvLocT"),
         ("fedproto", "standalone", "FPvStand"), ("fedtgp", "standalone", "TGPvStand"),
         ("fedtgp", "fedproto", "TGPvFP"), ("local_proto", "standalone", "LocvStand")]
PAIR_NAME = {"FPvLoc": "FedProto $-$ matched local", "TGPvLocT": "FedTGP $-$ matched local",
             "FPvStand": "FedProto $-$ standalone", "TGPvStand": "FedTGP $-$ standalone",
             "TGPvFP": "FedTGP $-$ FedProto", "LocvStand": "Matched local $-$ standalone"}
BAL_CODE = {"natural": "Nat", "balanced": "Bal"}
CONTRASTS = [  # (name, (endpoint, balance) minus (endpoint, balance),
    ("SevVsOne", ("sev3", "natural"), ("bin_t1", "natural"), r"granularity: 3-class $-$ binary $\theta{=}1$"),
    ("SevVsTwo", ("sev3", "natural"), ("bin_t2", "natural"), r"granularity: 3-class $-$ binary $\theta{=}2$"),
    ("TwoVsOne", ("bin_t2", "natural"), ("bin_t1", "natural"), r"threshold: $\theta{=}2$ $-$ $\theta{=}1$"),
    ("BalSev", ("sev3", "balanced"), ("sev3", "natural"), r"balance (3-class): balanced $-$ natural"),
    ("BalOne", ("bin_t1", "balanced"), ("bin_t1", "natural"), r"balance ($\theta{=}1$): balanced $-$ natural"),
    ("BalTwo", ("bin_t2", "balanced"), ("bin_t2", "natural"), r"balance ($\theta{=}2$): balanced $-$ natural"),
]
TBD = r"\tbd"


def sgn(x, d=3):
    """Signed number with a proper minus sign for LaTeX text mode."""
    return f"{'$-$' if x < 0 else '$+$'}{abs(x):.{d}f}"


class Fmt:
    def __init__(self, placeholder):
        self.ph = placeholder
    def f(self, x, d=3, sign=False):
        if self.ph:
            return TBD
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return "--"
        return sgn(x, d) if sign else f"{x:.{d}f}"
    def p(self, x):
        if self.ph:
            return TBD
        if x is None or np.isnan(x):
            return "--"
        return "<0.001" if x < 0.001 else f"{x:.3f}"


def select_final(runs, tune_seed=0):
    """Keep, for every learned method, only runs at the LR selected on validation (seed..."""
    runs = runs.copy()
    std = runs[runs.method == "standalone"]
    fl = runs[runs.method != "standalone"]
    if fl.empty:
        return std
    key = ["study", "endpoint", "balance", "partition", "method"]
    t = fl[fl.seed == tune_seed].groupby(key + ["lr"]).val_pooled.first().reset_index()
    best = t.loc[t.groupby(key).val_pooled.idxmax(), key + ["lr"]].rename(columns={"lr": "lr_sel"})
    fl = fl.merge(best, on=key)
    fl = fl[np.isclose(fl.lr.astype(float), fl.lr_sel.astype(float))].drop(columns="lr_sel")
    return pd.concat([std, fl], ignore_index=True)


def partition_table(final):
    """One score per study/endpoint/balance/method/partition/silo (mean over seeds) + 'all' =..."""
    g = (final.groupby(["study", "endpoint", "balance", "method", "partition", "silo"])
              .agg(f1=("test_f1", "mean"), f1_seed_sd=("test_f1", lambda v: np.std(v, ddof=1) if len(v) > 1 else 0.0),
                   n_seeds=("test_f1", "size"), n_train=("n_train", "mean"), n_test=("n_test", "mean"))
              .reset_index())
    allr = (g.groupby(["study", "endpoint", "balance", "method", "partition"])
              .agg(f1=("f1", "mean"), f1_seed_sd=("f1_seed_sd", "mean"), n_seeds=("n_seeds", "min"),
                   n_train=("n_train", "sum"), n_test=("n_test", "sum")).reset_index())
    allr["silo"] = "all"
    return pd.concat([g, allr], ignore_index=True)


def nb_corrected(d, ratio):
    """Nadeau & Bengio (2003) corrected resampled t-test on per-partition differences d."""
    d = np.asarray(d, float); R = len(d)
    if R < 2:
        return np.nan, np.nan, np.nan, np.nan
    v = np.var(d, ddof=1)
    se = np.sqrt((1.0 / R + ratio) * v)
    if se == 0:
        return d.mean(), d.mean(), np.nan, 1.0
    t = d.mean() / se
    q = stats.t.ppf(0.975, R - 1)
    return d.mean() - q * se, d.mean() + q * se, t, 2 * stats.t.sf(abs(t), R - 1)


def holm(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan)
    ok = ~np.isnan(p); idx = np.where(ok)[0]
    if len(idx) == 0:
        return out
    order = idx[np.argsort(p[idx])]; m = len(order); running = 0.0
    for k, i in enumerate(order):
        running = max(running, min(1.0, (m - k) * p[i]))
        out[i] = running
    return out


def compare(pt, study, endpoint, balance, a, b, silo):
    A = pt[(pt.study == study) & (pt.endpoint == endpoint) & (pt.balance == balance) & (pt.method == a) & (pt.silo == silo)]
    B = pt[(pt.study == study) & (pt.endpoint == endpoint) & (pt.balance == balance) & (pt.method == b) & (pt.silo == silo)]
    m = A.merge(B, on="partition", suffixes=("_a", "_b"))
    if m.empty:
        return None
    d = (m.f1_a - m.f1_b).to_numpy()
    ratio = float((m.n_test_a / m.n_train_a).mean())
    lo, hi, t, p_nb = nb_corrected(d, ratio)
    sd = np.std(d, ddof=1) if len(d) > 1 else np.nan
    t_plain, p_plain = (stats.ttest_rel(m.f1_a, m.f1_b) if len(d) > 1 and sd > 0 else (np.nan, np.nan))
    try:
        p_w = stats.wilcoxon(d, zero_method="wilcox", alternative="two-sided").pvalue if np.any(d != 0) else 1.0
    except ValueError:
        p_w = np.nan
    return dict(diff=d.mean(), sd=sd, lo=lo, hi=hi, t_corr=t, p_corr=p_nb, t=t_plain, p=p_plain,
                p_wilcoxon=p_w, dz=(d.mean() / sd if sd and sd > 0 else np.nan), R=len(d),
                n_pos=int((d > 0).sum()), n_neg=int((d < 0).sum()))


def all_comparisons(pt, endpoints):
    rows = []
    for (study, endpoint, balance), _ in pt.groupby(["study", "endpoint", "balance"]):
        if study != "main":
            continue
        silos = list(endpoints[endpoint]["silos"]) + ["all"]
        fam = []
        for a, b, code in PAIRS:
            for s in silos:
                r = compare(pt, study, endpoint, balance, a, b, s)
                if r is not None:
                    fam.append(dict(endpoint=endpoint, balance=balance, pair=code, a=a, b=b, silo=s, **r))
        if fam:
            pv = np.array([f["p_corr"] if f["silo"] != "all" else np.nan for f in fam])
            for f, ph in zip(fam, holm(pv)):
                f["p_holm"] = ph
            rows += fam
    return pd.DataFrame(rows)


def factor_analysis(pt):
    """Δ = FedProto − matched local per partition, for every (endpoint, balance) cell, plus..."""
    cells, deltas = [], {}
    for (endpoint, balance), _ in pt.groupby(["endpoint", "balance"]):
        for s in pt[pt.endpoint == endpoint].silo.unique():
            sub = pt[(pt.endpoint == endpoint) & (pt.balance == balance) & (pt.silo == s)]
            fp = sub[sub.method == "fedproto"].set_index("partition").f1
            lc = sub[sub.method == "local_proto"].set_index("partition").f1
            idx = fp.index.intersection(lc.index)
            if len(idx) == 0:
                continue
            d = (fp[idx] - lc[idx])
            ratio = float((sub.n_test / sub.n_train).mean())
            lo, hi, t, p = nb_corrected(d.to_numpy(), ratio)
            deltas[(endpoint, balance, s)] = (d, ratio)
            cells.append(dict(endpoint=endpoint, balance=balance, silo=s, delta=d.mean(), lo=lo, hi=hi, p_corr=p, R=len(d)))
    contr = []
    for name, c1, c2, desc in CONTRASTS:
        for s in ("eeg", "audio", "text", "all"):
            k1, k2 = (*c1, s), (*c2, s)
            if k1 in deltas and k2 in deltas:
                d1, r1 = deltas[k1]; d2, r2 = deltas[k2]
                idx = d1.index.intersection(d2.index)
                dd = (d1[idx] - d2[idx]).to_numpy()
                lo, hi, t, p = nb_corrected(dd, (r1 + r2) / 2)
                contr.append(dict(contrast=name, desc=desc, silo=s, diff=dd.mean(), lo=lo, hi=hi, p_corr=p, R=len(dd)))
    cdf = pd.DataFrame(contr)
    if not cdf.empty:
        cdf["p_holm"] = np.nan
        m = cdf.silo != "all"
        cdf.loc[m, "p_holm"] = holm(cdf.loc[m, "p_corr"].to_numpy())
    return pd.DataFrame(cells), cdf


def perclass_table(final, endpoint="sev3", methods=("local_proto", "fedproto")):
    rows = []
    sub = final[(final.study == "main") & (final.endpoint == endpoint) & (final.balance == "natural")]
    for m in methods:
        for s in ("eeg", "text", "audio"):
            q = sub[(sub.method == m) & (sub.silo == s)]
            if q.empty:
                continue
            per = []
            for (part, seed), r in q.groupby(["partition", "seed"]):
                y = np.array(json.loads(r.y.iloc[0])); p = np.array(json.loads(r.p.iloc[0]))
                per.append(f1_per_class(y, p) + [np.bincount(y, minlength=3)])
            f = np.array([x[:3] for x in per]); sup = np.mean([x[3] for x in per], axis=0)
            rows.append(dict(method=m, silo=s, f0=f[:, 0].mean(), f1=f[:, 1].mean(), f2=f[:, 2].mean(),
                             f0sd=f[:, 0].std(ddof=1), f1sd=f[:, 1].std(ddof=1), f2sd=f[:, 2].std(ddof=1),
                             sup="/".join(f"{v:.1f}" for v in sup)))
    return pd.DataFrame(rows)


def f1_per_class(y, p):
    from sklearn.metrics import f1_score
    return list(f1_score(y, p, labels=[0, 1, 2], average=None, zero_division=0))


def _ms(F, pt, study, ep, bal, m, s):
    v = pt[(pt.study == study) & (pt.endpoint == ep) & (pt.balance == bal) & (pt.method == m) & (pt.silo == s)].f1
    if F.ph:
        return TBD
    return "--" if v.empty else f"{v.mean():.3f}\\,$\\pm$\\,{(v.std(ddof=1) if len(v) > 1 else 0):.3f}"


def table_endpoint(F, pt, ep, endpoints, methods):
    silos = list(endpoints[ep]["silos"]) + ["all"]
    cols = "l" + "c" * len(methods)
    L = [rf"\begin{{tabular}}{{{cols}}}", r"\toprule",
         r"\textbf{Silo} & " + " & ".join(rf"\textbf{{{M_SHORT[m]}}}" for m in methods) + r" \\", r"\midrule"]
    for s in silos:
        if s == "all":
            L.append(r"\midrule")
        L.append(f"{S_NAME[s]} & " + " & ".join(_ms(F, pt, "main", ep, "natural", m, s) for m in methods) + r" \\")
    L += [r"\bottomrule", r"\end{tabular}"]
    return "\n".join(L)


def table_stats(F, cmp, ep, pairs=("FPvLoc", "TGPvLocT", "FPvStand", "TGPvStand", "TGPvFP")):
    L = [r"\begin{tabular}{llccccc}", r"\toprule",
         r"\textbf{Comparison} & \textbf{Silo} & \textbf{$\Delta$ [95\% CI]} & \textbf{$d_z$} & "
         r"\textbf{$p_{\mathrm{corr}}$} & \textbf{$p_{\mathrm{Holm}}$} & \textbf{$p_{\mathrm{W}}$} \\", r"\midrule"]
    sub = cmp[cmp.endpoint == ep] if not cmp.empty else cmp
    first = True
    for code in pairs:
        q = sub[(sub.pair == code) & (sub.silo != "all")] if not sub.empty else sub
        if q.empty:
            continue
        if not first:
            L.append(r"\midrule")
        first = False
        for k, (_, r) in enumerate(q.iterrows()):
            ci = TBD if F.ph else f"{sgn(r['diff'])} [{sgn(r.lo)}, {sgn(r.hi)}]"
            L.append(f"{PAIR_NAME[code] if k == 0 else ''} & {S_NAME[r.silo]} & {ci} & {F.f(r.dz, 2, True)} & "
                     f"{F.p(r.p_corr)} & {F.p(r.p_holm)} & {F.p(r.p_wilcoxon)} \\\\")
    L += [r"\bottomrule", r"\end{tabular}"]
    return "\n".join(L)


def table_factor(F, cells, contr):
    L = [r"\begin{tabular}{llccc}", r"\toprule",
         r"\textbf{Endpoint} & \textbf{Training balance} & \textbf{EEG} & \textbf{Audio} & \textbf{Text} \\", r"\midrule"]
    for ep in ("sev3", "bin_t1", "bin_t2"):
        for bal in ("natural", "balanced"):
            vals = []
            for s in ("eeg", "audio", "text"):
                q = cells[(cells.endpoint == ep) & (cells.balance == bal) & (cells.silo == s)] if not cells.empty else cells
                vals.append(TBD if F.ph else ("--" if q.empty else f"{sgn(q['delta'].iloc[0])} [{sgn(q.lo.iloc[0])}, {sgn(q.hi.iloc[0])}]"))
            L.append(f"{EP_SHORT[ep] if bal == 'natural' else ''} & {bal} & " + " & ".join(vals) + r" \\")
    L += [r"\midrule", r"\multicolumn{5}{l}{\textit{Contrasts of $\Delta$ (one factor changed)}} \\"]
    for name, c1, c2, desc in CONTRASTS:
        vals = []
        for s in ("eeg", "audio", "text"):
            q = contr[(contr.contrast == name) & (contr.silo == s)] if not contr.empty else contr
            if F.ph:
                vals.append(TBD)
            elif q.empty:
                vals.append("--")
            else:
                star = "$^{*}$" if q.p_holm.iloc[0] < 0.05 else ""
                vals.append(f"{sgn(q['diff'].iloc[0])} [{sgn(q.lo.iloc[0])}, {sgn(q.hi.iloc[0])}]{star}")
        L.append(rf"\multicolumn{{2}}{{l}}{{{desc}}} & " + " & ".join(vals) + r" \\")
    L += [r"\bottomrule", r"\end{tabular}"]
    return "\n".join(L)


def table_perclass(F, pc):
    L = [r"\begin{tabular}{llcccc}", r"\toprule",
         r"\textbf{Method} & \textbf{Silo} & \textbf{None-min.} & \textbf{Mild-mod.} & \textbf{Severe} & \textbf{Test support} \\",
         r"\midrule"]
    for m in ("local_proto", "fedproto"):
        for k, silo in enumerate(("eeg", "text", "audio")):
            q = pc[(pc.method == m) & (pc.silo == silo)] if not pc.empty else pc
            lead = M_SHORT[m] if k == 0 else ""
            if F.ph or q.empty:
                L.append(f"{lead} & {S_NAME[silo]} & {TBD} & {TBD} & {TBD} & {TBD} \\\\")
            else:
                r = q.iloc[0]
                L.append(f"{lead} & {S_NAME[silo]} & {r.f0:.3f}\\,$\\pm$\\,{r.f0sd:.3f} & "
                         f"{r.f1:.3f}\\,$\\pm$\\,{r.f1sd:.3f} & {r.f2:.3f}\\,$\\pm$\\,{r.f2sd:.3f} & {r.sup} \\\\")
        if m == "local_proto":
            L.append(r"\midrule")
    L += [r"\bottomrule", r"\end{tabular}"]
    return "\n".join(L)


def table_individual(F, pt, endpoints, methods):
    """Supplementary longtable: per-partition macro-F1 (mean over seeds) for every configuration."""
    P = sorted(pt.partition.unique())
    L = [r"\setlength{\tabcolsep}{2.5pt}", r"\begin{longtable}{lll" + "c" * len(P) + "cc}",
         r"\caption{Individual results for every configuration: test macro-F1 per subject-level partition "
         r"(mean over seeds within the partition), with mean and SD across partitions. Per-seed values are in "
         r"the accompanying \texttt{runs\_long.csv}.}\label{tab:individual}\\", r"\toprule",
         r"\textbf{Endpoint} & \textbf{Method} & \textbf{Silo} & " + " & ".join(rf"\textbf{{P{p + 1}}}" for p in P)
         + r" & \textbf{Mean} & \textbf{SD} \\", r"\midrule", r"\endfirsthead", r"\toprule",
         r"\textbf{Endpoint} & \textbf{Method} & \textbf{Silo} & " + " & ".join(rf"\textbf{{P{p + 1}}}" for p in P)
         + r" & \textbf{Mean} & \textbf{SD} \\", r"\midrule", r"\endhead"]
    for ep in [e for e in EP_CODE if e in pt.endpoint.unique()]:
        for m in methods:
            for s in endpoints[ep]["silos"]:
                q = pt[(pt.study == "main") & (pt.endpoint == ep) & (pt.balance == "natural") & (pt.method == m) & (pt.silo == s)].set_index("partition").f1
                vals = [F.f(q.get(p, np.nan), 2) for p in P]
                L.append(f"{EP_CODE[ep]} & {M_SHORT[m]} & {S_NAME[s]} & " + " & ".join(vals)
                         + f" & {F.f(q.mean() if len(q) else np.nan)} & {F.f(q.std(ddof=1) if len(q) > 1 else np.nan)} \\\\")
        L.append(r"\midrule")
    L[-1] = r"\bottomrule"
    L.append(r"\end{longtable}")
    return "\n".join(L)


def write_macros(F, pt, cmp, cells, contr, path, n_seeds, endpoints):
    M = {}
    M["Rnpart"] = str(pt.partition.nunique()) if not F.ph else TBD
    M["Rnseeds"] = str(int(pt.n_seeds.max())) if not F.ph else TBD
    for ep in EP_CODE:
        if ep not in endpoints:
            continue
        for m in M_CODE:
            for s in list(endpoints[ep]["silos"]) + ["all"]:
                v = pt[(pt.study == "main") & (pt.endpoint == ep) & (pt.balance == "natural") & (pt.method == m) & (pt.silo == s)].f1
                M[f"R{EP_CODE[ep]}{M_CODE[m]}{S_CODE[s]}Mean"] = F.f(v.mean() if len(v) else np.nan)
                M[f"R{EP_CODE[ep]}{M_CODE[m]}{S_CODE[s]}SD"] = F.f(v.std(ddof=1) if len(v) > 1 else np.nan)
        for a, b, code in PAIRS:
            for s in list(endpoints[ep]["silos"]) + ["all"]:
                q = cmp[(cmp.endpoint == ep) & (cmp.pair == code) & (cmp.silo == s)] if not cmp.empty else cmp
                r = q.iloc[0] if len(q) else None
                base = f"C{EP_CODE[ep]}{code}{S_CODE[s]}"
                M[base + "Diff"] = F.f(r["diff"] if r is not None else np.nan, 3, True)
                M[base + "Lo"] = F.f(r.lo if r is not None else np.nan, 3, True)
                M[base + "Hi"] = F.f(r.hi if r is not None else np.nan, 3, True)
                M[base + "P"] = F.p(r.p_corr if r is not None else np.nan)
                M[base + "Padj"] = F.p(r.p_holm if r is not None and "p_holm" in r else np.nan)
                M[base + "Pw"] = F.p(r.p_wilcoxon if r is not None else np.nan)
                M[base + "Dz"] = F.f(r.dz if r is not None else np.nan, 2, True)
                M[base + "Npos"] = TBD if F.ph else (str(r.n_pos) if r is not None else "--")
    for ep in ("sev3", "bin_t1", "bin_t2"):
        for bal in ("natural", "balanced"):
            for s in ("eeg", "audio", "text", "all"):
                q = cells[(cells.endpoint == ep) & (cells.balance == bal) & (cells.silo == s)] if not cells.empty else cells
                r = q.iloc[0] if len(q) else None
                base = f"F{EP_CODE[ep]}{BAL_CODE[bal]}{S_CODE[s]}"
                M[base + "Delta"] = F.f(r["delta"] if r is not None else np.nan, 3, True)
                M[base + "Lo"] = F.f(r.lo if r is not None else np.nan, 3, True)
                M[base + "Hi"] = F.f(r.hi if r is not None else np.nan, 3, True)
    for name, *_ in CONTRASTS:
        for s in ("eeg", "audio", "text", "all"):
            q = contr[(contr.contrast == name) & (contr.silo == s)] if not contr.empty else contr
            r = q.iloc[0] if len(q) else None
            base = f"Fc{name}{S_CODE[s]}"
            M[base + "Diff"] = F.f(r["diff"] if r is not None else np.nan, 3, True)
            M[base + "Lo"] = F.f(r.lo if r is not None else np.nan, 3, True)
            M[base + "Hi"] = F.f(r.hi if r is not None else np.nan, 3, True)
            M[base + "P"] = F.p(r.p_holm if r is not None else np.nan)
    with open(path, "w") as fh:
        fh.write("% auto-generated by 3_FL_experiments.ipynb -- do not edit by hand\n")
        for k, v in M.items():
            assert k.isalpha(), k
            fh.write(f"\\newcommand{{\\{k}}}{{{v}}}\n")
    return M


def write_hparam_table(hp, gate_target, lr_grid, path, lr_counts=None):
    lrs = ", ".join(f"{v:g}" for v in lr_grid)
    rows = [
        ("Projection head $f_s$", rf"$d_s\!\to\!{hp['HIDDEN_DIM']}\!\to\!{hp['HIDDEN_DIM']}\!\to\!{hp['SHARED_DIM']}$; "
                                  rf"Linear--LayerNorm--GELU--Dropout({hp['HEAD_DROPOUT']}) blocks; output L2-normalised"),
        ("Classifier head", rf"${hp['SHARED_DIM']}\!\to\!{hp['HIDDEN_DIM']}\!\to\!K$, LayerNorm, GELU, Dropout({hp['HEAD_DROPOUT']}); local, never shared"),
        ("Input standardisation", "z-score per silo with training-split statistics"),
        ("Optimiser", rf"AdamW, weight decay $10^{{-4}}$, gradient clipping {hp['GRAD_CLIP']}"),
        ("Learning rate", rf"selected per partition and method from \{{{lrs}\}} on pooled validation macro-F1"),
        ("Rounds / local epochs / batch", rf"{hp['N_ROUNDS']} / {hp['LOCAL_EPOCHS']} / {hp['BATCH_SIZE']}; early stop after {hp['EARLY_STOP_PATIENCE']} rounds without validation gain"),
        ("Prototype loss", rf"temperature $\tau={hp['PROTO_TEMPERATURE']}$, inverse-frequency class weights"),
        ("Compactness term", rf"$\lambda_{{\mathrm{{comp}}}}={hp['COMPACT_LAMBDA']}$, supervised-contrastive temperature 0.07"),
        ("Classification loss", rf"focal, $\gamma={hp['FOCAL_GAMMA']}$, weight {hp['CE_WEIGHT']}, inverse-frequency class weights"),
        ("Augmentation", rf"Gaussian embedding noise $\sigma={hp['AUGMENT_NOISE_STD']}$; mixup $\alpha={hp['MIXUP_ALPHA']}$ (classifier path only)"),
        ("FedTGP server", rf"margin $m={hp['TGP_MARGIN']}$ (mean over competing prototypes), Adam lr $10^{{-3}}$, one step per round"),
        ("Reliability weights $r_k^{(s)}$", "per-class validation F1 of nearest-prototype prediction"),
        ("Gate initial value $g_s^{(0)}$", ", ".join(f"{k.upper() if k=='eeg' else k.capitalize()} {v:g}" for k, v in hp["GATE_INIT"].items())),
        ("Gate target", "; ".join(
            (f"{k.upper() if k=='eeg' else k.capitalize()} $\\min({m:g}\\,\\bar r_s,{c:g})$" if c is not None
             else f"{k.upper() if k=='eeg' else k.capitalize()} $\\bar r_s$") for k, (m, c) in gate_target.items())
         + rf"; $\bar r_s$ clipped to [{hp['GATE_MIN']}, {hp['GATE_MAX']}]; EMA momentum {hp['GATE_ADAPT_MOMENTUM']}"),
        ("Binary decision threshold", rf"tuned per silo on validation macro-F1 over {hp['THRESHOLD_GRID']} values in [0.05, 0.95]"),
    ]
    L = [r"\begin{tabular}{p{0.30\textwidth}p{0.64\textwidth}}", r"\toprule",
         r"\textbf{Component} & \textbf{Setting} \\", r"\midrule"]
    L += [f"{a} & {b} \\\\" for a, b in rows]
    L += [r"\bottomrule", r"\end{tabular}"]
    open(path, "w").write("\n".join(L))


def export_all(runs, out, endpoints, placeholder=False, tune_seed=0):
    os.makedirs(out, exist_ok=True)
    F = Fmt(placeholder)
    final = select_final(runs, tune_seed)
    pt = partition_table(final)
    cmp = all_comparisons(pt, endpoints)
    cells, contr = factor_analysis(pt)
    pc = perclass_table(final)
    if not placeholder:
        pt.to_csv(f"{out}/per_partition_scores.csv", index=False)
        cmp.to_csv(f"{out}/comparisons.csv", index=False)
        cells.to_csv(f"{out}/factor_cells.csv", index=False)
        contr.to_csv(f"{out}/factor_contrasts.csv", index=False)
        pc.to_csv(f"{out}/perclass_sev3.csv", index=False)
    meth5 = ["standalone", "local_proto", "fedproto", "local_tgp", "fedtgp"]
    present = set(pt.endpoint.unique())
    for ep in ("bin_ea", "bin_t1", "bin_t2", "sev3", "bin_mixed"):
        if ep in present:
            open(f"{out}/tab_{ep}.tex", "w").write(table_endpoint(F, pt, ep, endpoints, meth5))
            open(f"{out}/tab_stats_{ep}.tex", "w").write(table_stats(F, cmp, ep))
    open(f"{out}/tab_factor.tex", "w").write(table_factor(F, cells, contr))
    open(f"{out}/tab_perclass_sev3.tex", "w").write(table_perclass(F, pc))
    open(f"{out}/tab_individual.tex", "w").write(table_individual(F, pt, endpoints, meth5))
    write_macros(F, pt, cmp, cells, contr, f"{out}/results_macros.tex", None, endpoints)
    return pt, cmp, cells, contr

In [ ]:
if RUN_EXPORT:
    parts_csv = [f"{R1_OUT}/runs_long.csv", f"{R1_OUT}/runs_long_B.csv"]
    missing = [p for p in parts_csv if not os.path.exists(p)]
    assert not missing, f"missing results file(s): {missing}"
    runs = pd.concat([pd.read_csv(p) for p in parts_csv], ignore_index=True)
    runs = runs.drop_duplicates(subset=["study", "endpoint", "balance", "partition", "method", "lr", "seed", "silo"])
    runs.to_csv(f"{R1_OUT}/runs_long_all.csv", index=False)
    print("combined runs:", len(runs), "rows ->", f"{R1_OUT}/runs_long_all.csv")
    print(runs.groupby(["study", "endpoint", "balance", "method"]).partition.nunique().unstack())
    export_all(runs, LATEX_OUT, endpoints=ENDPOINTS, placeholder=False, tune_seed=TUNE_SEED)
    print("LaTeX outputs ->", LATEX_OUT)
    for f in sorted(os.listdir(LATEX_OUT)):
        print("  ", f)
else:
    print("RUN_EXPORT = False -> export skipped (set True in SESSION A after both sessions finish)")

## 10 · Executed configuration and hyper-parameter table

In [ ]:
if RUN_EXPORT:
    executed = dict(BASE_HP=BASE_HP, GATE_TARGET=GATE_TARGET, LR_GRID=LR_GRID, TUNE_LR=TUNE_LR, N_SEEDS=N_SEEDS,
                    EEG_SOURCE=EEG_SOURCE, AUDIO_LAYER_MODE=AUDIO_LAYER_MODE, ENDPOINTS=ENDPOINTS,
                    partitions=ALL_PARTITIONS, torch=torch.__version__)
    json.dump(executed, open(f"{R1_OUT}/executed_config.json", "w"), indent=2, default=str)
    _final = select_final(runs, TUNE_SEED)
    sel = (_final[(_final.method != "standalone") & (_final.study == "main")]
           .groupby(["endpoint", "partition", "method"]).lr.first().reset_index())
    sel.to_csv(f"{R1_OUT}/selected_lr_per_partition.csv", index=False)
    print(sel.groupby(["endpoint", "method"]).lr.value_counts().unstack(fill_value=0))
    if AUDIO_LAYER_MODE == "per_partition":
        al = runs[runs.silo == "audio"].groupby("partition").audio_layer.first()
        print("audio layer per partition:", al.to_dict())
    write_hparam_table(BASE_HP, GATE_TARGET, LR_GRID, f"{LATEX_OUT}/tab_hparams.tex",
                       lr_counts=sel.lr.value_counts().to_dict())
    print("saved executed_config.json, selected_lr_per_partition.csv, tab_hparams.tex")